# **Dissolved Gas Concentrations and Headspace Equilibrium**

### Environmental Applications
Environmental scientists often seek to gather as many potential variables from their study system as possible to maintain the highest degree of accuracy in modeling. For many applications, it is desirable for them to understand gas production and dynamics. Examples include measuring CO<sub>2</sub> flux rates from forests or CH<sub>4</sub> concentrations in aquatic environments (see this [USGS](https://www.sciencedirect.com/topics/biochemistry-genetics-and-molecular-biology/headspace-technique) article discussing fascinating application for this technique). However, when trying to answer questions such as these, measuring the outflow of these gases from their origins only paints part of the story. In cases like this, scientists may want to know the gas concentrations actively within their subject matrix (i.e., soil, water, etc.). 

### Foundations and Henry's Law
This gave birth to a quantitative method called [headspace equilibrium](https://www.sciencedirect.com/topics/biochemistry-genetics-and-molecular-biology/headspace-technique) in the late 1960s, which allowed scientists to determine gas concentrations in aqeuous solutions specifically. This technique was built on a physical chemistry principle established over a century prior called [Henry's Law](https://chem.libretexts.org/Bookshelves/Physical_and_Theoretical_Chemistry_Textbook_Maps/Supplemental_Modules_(Physical_and_Theoretical_Chemistry)/Physical_Properties_of_Matter/Solutions_and_Mixtures/Ideal_Solutions/Dissolving_Gases_In_Liquids_Henry%27s_Law), which states that "At a constant temperature, the amount of a given gas that dissolves in a given type and volume of liquid is directly proportional to the partial pressure of that gas in equilibrium with that liquid." In other words, the gas concentration in the liquid solution is direclty proportional to the partial pressure above the liquid. This, as is often the case regarding laws of the natural world, is founded in the idea that all systems inherently seek equilibrium. If you would like a deeper explanation of how Henry's Law functions, take a peek at this [video](https://www.youtube.com/watch?v=9JtTpPEesOk) which utilizes the helpful graphic below.

<div>
<img src="images/henrysLaw.png" width = "750"/>
</div>

## Sampling Technique and Workflow
The actual sampling process with this method is actually quite straight forward. A generalized approach is listed in order below:
- Start by taking a water sample and sealing it in an air-tight bottle.
- Remove a known volume of liquid from the sample bottle and insert it into a completely evacuated vial.
- Fill the remaining volume in that vial with a known gas (could be ambient air, could be something else like nitrogen).
- Once sealed, shake the vial for up to 10 minutes then let it rest for a bit to ensure equilibrium has taken effect.
- Draw a gas sample from the equilibrated headspace and measure it using some sort of gas analyzer or chromatograph.
- Finally, use Henry's Law to back calculate the gas concentration in the liquid given recorded parameters like liquid volume, headspace volume and temperature.

For most applications, the pressure can be assumed as 1 atm unless you are at unreasonably high elevations. Additionally, it is important to try and maintain a constant temperature throughout this process because temperature can influence the solubility potential of gases and liquids. 

<div>
<img src="images/headEqu.png" width = "650"/>
</div>

## Let's try this out!
We're going to create a dataframe that represents the measurements from our water sample. We're going to assume that you have already conducted the headscape equilibrium and thus have a corresponding gas concentration (we'll use CO<sub>2</sub> for this example). Follow along with the code below to see how you would apply Henry's Law to determine gas concentrations in an aqueous solution.

In [3]:
# Create a data set for our water sample - let's call it 'water_sample'
water_sample <- data.frame(
    site_id = as.character("big_lake"),
    sample_date = as.POSIXct("05-05-2026 08:00:00"),
    tempC = as.numeric(22.5),
    co2_ppm = as.numeric(8.3))

In [4]:
# Let's check we successfully created the data frame we want
head(water_sample)

# Use head() at any point during this exercise to make sure modifications to the water_sample data frame actually took effect

,site_id,sample_date,tempC,co2_ppm
,<chr>,<dttm>,<dbl>,<dbl>
1,big_lake,5-05-20,22.5,8.3


In [5]:
# Now that we have a data frame, we want to start by converting temperature from celsius to Kelvin
water_sample$tempK <- water_sample$tempC + 273.15

In [6]:
# Next, we are going to correct, modify and generate a dimensional unit for Henry's Law constant so that it is 
# compatible with the rest of the equation. 

kH = (2.7182818^(-58.0931+(90.5069*(100/water_sample$tempK))+(22.294*log(water_sample$tempK/100))))*((0.0821*273.15)+
        ((-1636.75+(12.0408*273.15)-(3.27957*0.01*273.15*273.15)+(3.16528*0.00001*273.15*273.15*273.15))/1000))

#### A Note: Manufacturing Henry's Law Constant
There are several corrections and alterations that need to be made in order to transform the normally dimensionless and unitless Henry's Law Constant into a tangible variable we can fit into our equation. Weiss (1974) published a multitude of equations and values for this exact purpose as it relates to CO<sub>2</sub>. These corrections account for things like salinity, pressure, thermodynamics and molar volume as they relate to CO<sub>2</sub>. If you would like to read more about the derivation of these values, check out R.F. Weiss' 1974 paper [Carbon Dioxide in Water and Seawater: The Solubility of a Non-ideal Gas](https://www.sciencedirect.com/science/article/pii/0304420374900152).

In [7]:
# We need to assign the rest of the variables needed for this equation. We are going to assume normal atmospheric pressure

bp <- 1
liquid_volume_L <- 0.08
headspace_volume_L <- 0.02

#### Now the fun part - using Henry's Law to actually back calculate some values!
But first, a few variable definitions:
- LGaE = liquid gas concentration after equilibrium
- HGaE = headspace gas concentration after equilibrium
- TGiS = total gas concentration in system
- DCiL = original dissolved gas concentration in liquid (what we are ultimately solving for)

Additionally, we are using Henry's Law hand-in-hand with the Ideal Gas Law, which is described below. The two equations complement one another: Henry's Law gives us the ratio between dissolved gas concentration and the partial pressure in the system while the Ideal Gas Law allows us to convert units from ppm (parts per million) to molar concentrations (umol/L). In the cell block below, we recongifure the Ideal Gas Law to now state ***n/V = P/RT*** because we are looking for umol/L of CO<sub>2</sub>. For the context of this exercise, we are using 0.0821 as our ideal gas constant. 

<u>**Ideal Gas Law**</u>  
PV = nRT

- P = pressure 
- V = volume
- n = molar mass
- R = ideal gas constant
- T = temperature

In [8]:
# Find the gas concentrations for headspace then use Henry's Law to infer what it is in the liquid (i.e., law of equilibrium)
HGaE = water_sample$co2_ppm/(0.0821*water_sample$tempK)
LGaE = (water_sample$co2_ppm * kH * bp)/(0.0821*water_sample$tempK)

In [15]:
# Now calculate the total gas concentration from the system as a whole using a mass balance approach
TGiS = (HGaE*headspace_volume_L)+(LGaE*liquid_volume_L)

In [16]:
# Lastly, divide that total gas concentration by the volume of the liquid to retrieve the original dissolved concentration
DCiL = TGiS/liquid_volume_L

# Add it to your data frame!
water_sample$co2_umolL<-as.numeric(DCiL)

In [17]:
# Check your data frame to make sure the final result is there (units are umol/L)
head(water_sample)

,site_id,sample_date,tempC,co2_ppm,tempK,co2_umolL
,<chr>,<dttm>,<dbl>,<dbl>,<dbl>,<dbl>
1,big_lake,5-05-20,22.5,8.3,295.65,0.3626884
